In [1]:
import pandas as pd
import numpy as np
import sys

#Data extraction directly from DfE SEN2 publisehd data sets
#NEEDS TO BE UPDATED AFTER EACH RELEASE
requests = pd.read_csv(
    "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/2cb3f1e5-7961-4034-9643-0adb38a1fefa/csv"
)
assessments = pd.read_csv(
    "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/96aff67b-a51f-4982-97ce-b720006edf71/csv"
)

In [ ]:
#Definining Functions

weights = np.array([1, 2, 3])
def weighted_avg_last3(series):
    """
    Calculate a weighted average of the last up to three non-missing values
    in a pandas Series, using predefined ascending weights.

    This function is typically used for forecasting or smoothing where
    recent observations should contribute more heavily than older ones.
    Missing values (NaN) are ignored. If fewer than three valid values
    are available, all available values are used with the corresponding
    subset of weights.

    Parameters
    ----------
    series : pandas.Series
        A numeric pandas Series from which the last values will be used.

    Returns
    -------
    float
        The weighted average of the last (up to) three valid values.
        Returns NaN if no valid values exist.

    Notes
    -----
    - The weighting scheme is fixed as [1, 2, 3], with 3 applied to the
      most recent value.
    - The function automatically aligns the weight vector to the number
      of available values.
    - Designed for use where more recent values should have greater
      influence on the result.
    """
    vals = series.dropna()
    if len(vals) == 0:
        return np.nan

    last_vals = vals.tail(3).values
    w = weights[-len(last_vals):]

    return np.average(last_vals, weights=w)



def tame_outliers(
    group: pd.DataFrame,
    value_col: str,
    out_col: str,
    k: float = 2.5,
    global_clip=(0, 100),
) -> pd.DataFrame:
    """
    Apply robust outlier taming to a numeric column within a grouped DataFrame.

    This function uses the Median Absolute Deviation (MAD) method to detect
    and tame outliers at the LA level. It is designed for short, noisy
    year-on-year time series such as % change, % refuse-to-assess (R2A),
    or % refuse-to-issue (R2I). The goal is to prevent extreme or anomalous
    values from disproportionately affecting trend-based forecasting.

    Parameters
    ----------
    group : pandas.DataFrame
        A subset of the full dataset representing a single LA group
        (usually produced by df.groupby("old_la_code").apply(...)).
    value_col : str
        Name of the numeric column to be evaluated and tamed.
    out_col : str
        Name of the new column to store the tamed values.
    k : float, optional (default=2.5)
        The scaling factor applied to the MAD-based robust z-score.
        Higher values = fewer points classified as outliers.
    global_clip : tuple, optional (default=(0, 100))
        Hard bounds applied to the output values regardless of MAD outcome.
        Typically used for percentage data.

    Returns
    -------
    pandas.DataFrame
        A copy of the input group with:
        - a new column ``out_col`` containing the tamed values
        - a boolean column ``out_col + "_is_outlier"`` indicating whether
          the original value was considered an outlier before taming.

    Notes
    -----
    - If fewer than 3 non-missing values are available, MAD cannot be
      reliably computed, so values are simply clipped to ``global_clip``.
    - MAD is preferred over standard deviation for short or skewed data,
      as it is more robust to extreme values.
    - The constant 1.4826 scales MAD to be comparable to standard deviation
      for normally distributed data.
    """
    g = group.copy()
    y = g[value_col]

    if y.notna().sum() < 3:
        g[out_col] = y.clip(*global_clip)
        g[f"{out_col}_is_outlier"] = False
        return g

    med = y.median(skipna=True)
    mad = (y - med).abs().median(skipna=True)

    if mad == 0 or np.isnan(mad):
        lower, upper = global_clip
    else:
        scale = 1.4826 * mad
        lower = max(med - k * scale, global_clip[0])
        upper = min(med + k * scale, global_clip[1])

    g[out_col] = y.clip(lower, upper)
    g[f"{out_col}_is_outlier"] = (y < lower) | (y > upper)
    return g


def linear_trend_by_column(
    df: pd.DataFrame,
    value_col: str,
    group_col: str = "old_la_code",
    time_col: str = "time_period",
    latest_aux_cols=("requests_received_in_year",),
) -> pd.DataFrame:
    """
    Fit a per-group linear trend (y = a*x + b) and project the next value.

    This utility groups the input DataFrame by ``group_col``, sorts each group
    by ``time_col``, and fits a simple least-squares linear regression to
    ``value_col`` (after dropping missing values). For each group it returns:
    the latest observed value, slope and intercept of the fitted line, and a
    one-step-ahead projection (next year/period). It also returns the latest
    non-missing values for any auxiliary columns listed in ``latest_aux_cols``.

    The function is designed for short annual series such as YoY % metrics,
    R2A/R2I rates (after any taming you apply), or other trend-like indicators.

    Parameters
    ----------
    df : pandas.DataFrame
        Input data containing at least ``group_col``, ``time_col``, and
        ``value_col``.
    value_col : str
        The name of the numeric column to model and project (dependent variable).
    group_col : str, optional
        Column that identifies groups (e.g., local authorities). Default is
        ``"old_la_code"``.
    time_col : str, optional
        Column representing the time axis used for regression (e.g., year).
        Must be numeric or castable to float. Default is ``"time_period"``.
    latest_aux_cols : tuple of str, optional
        Names of additional columns for which the latest non-missing value per
        group should be returned. Each will appear in the output as
        ``"latest_<colname>"``. Default is ``("requests_received_in_year",)``.

    Returns
    -------
    pandas.DataFrame
        One row per group with the following columns:
        - ``group_col``: group identifier (e.g., LA code)
        - ``latest_value``: latest non-missing value of ``value_col`` in the group
        - ``linear_slope_per_year``: slope (a) of the fitted line
        - ``linear_intercept``: intercept (b) of the fitted line
        - ``linear_proj_next_year``: projection at ``max(time_col) + 1``
        - ``latest_<aux>``: latest non-missing values for each auxiliary column
          listed in ``latest_aux_cols``

        If a group has fewer than two non-missing observations in ``value_col``,
        slope/intercept/projection are returned as NaN, and ``latest_value`` is
        provided when available.

    Notes
    -----
    - The regression is computed via ``numpy.polyfit(x, y, 1)`` on each group.
    - ``time_col`` is cast to float before fitting; ensure it has a meaningful
      numeric scale (e.g., calendar years).
    - This function does **not** clip projected values; if you are modelling
      bounded percentages, consider clipping after the call.
    """
    df = df.sort_values([group_col, time_col]).copy()
    results = []

    for la, g in df.groupby(group_col):
        sub = g.dropna(subset=[value_col])
        latest_year = g[time_col].max()

        # collect latest values for requested auxiliary columns
        latest_vals = {}
        for c in latest_aux_cols:
            latest_vals[f"latest_{c}"] = (
                g[c].dropna().iloc[-1]
                if c in g.columns and g[c].notna().any()
                else np.nan
            )

        # not enough points to fit a line
        if sub.shape[0] < 2:
            results.append(
                {
                    group_col: la,
                    "latest_value": (
                        sub[value_col].dropna().iloc[-1]
                        if sub[value_col].notna().any()
                        else np.nan
                    ),
                    "linear_slope_per_year": np.nan,
                    "linear_intercept": np.nan,
                    "linear_proj_next_year": np.nan,
                    **latest_vals,
                }
            )
            continue

        x = sub[time_col].astype(float).values
        y = sub[value_col].astype(float).values
        slope, intercept = np.polyfit(x, y, 1)
        next_year = float(latest_year) + 1.0
        proj_next = slope * next_year + intercept

        results.append(
            {
                group_col: la,
                "latest_value": sub[value_col].dropna().iloc[-1],
                "linear_slope_per_year": slope,
                "linear_intercept": intercept,
                "linear_proj_next_year": proj_next,
                **latest_vals,
            }
        )

    return pd.DataFrame(results)

# adding behaviour metrics
summary_cols = ["avg_pct_change", "weighted_pct_change", "pct_change"]
def row_std_proxy(row):
    """
    Compute a volatility proxy for a single row by calculating the standard
    deviation across selected summary percentage columns.

    This helper function is used to quantify how variable or unstable an
    LA's historic metrics are by taking the standard deviation of all
    non-missing values across a predefined set of summary columns
    (e.g., YoY % change, weighted % change, latest % change).

    The function returns NaN if fewer than two valid values are available,
    because a standard deviation cannot be meaningfully computed in that case.

    Parameters
    ----------
    row : pandas.Series
        A row from the DataFrame being evaluated. It must contain all columns
        listed in the global variable ``summary_cols``.

    Returns
    -------
    float
        The standard deviation of the non-missing values across ``summary_cols``,
        expressed as a float. Returns NaN when fewer than two valid values exist.

    Notes
    -----
    - ``summary_cols`` must be defined in the surrounding scope (e.g., a list
      such as ``["avg_pct_change", "weighted_pct_change", "pct_change"]``).
    - Standard deviation is computed with ``ddof=0`` for population variance.
    - This metric is used to decide whether an LA's behaviour is "volatile" in
      the forecasting model selection logic.
    """
    vals = [row[c] for c in summary_cols if pd.notna(row[c])]
    return float(np.nanstd(vals, ddof=0)) if len(vals) >= 2 else np.nan



def recommend_model_row(row):
    """
    Recommend an appropriate forecasting model for a single local authority (LA)
    based on behavioural indicators derived from historic data.

    This function evaluates several row-level metrics—volatility, trend strength,
    missing data, and the magnitude of recent shocks—to determine which of the
    four forecasting models is most suitable for the LA:

        • Option 1 – Crude Average
        • Option 2 – Last-Year Only
        • Option 3 – Weighted Last 3 Years
        • Option 4 – Linear Trend

    The logic prioritises robustness by steering high-volatility or strongly
    trending LAs toward a trend-based forecast, while stable LAs are directed
    toward weighted or average-based models. A large discrepancy between the
    last-year value and the weighted 3-year average triggers a Last-Year Only
    recommendation.

    Parameters
    ----------
    row : pandas.Series
        A single row from the model-selection table containing:
        - volatility_proxy : float
            Standard deviation of historic % values, used to assess stability.
        - linear_slope_per_year : float
            Fitted slope from the linear trend model; measures direction and strength.
        - missing_years : int
            Count of missing data points in the LA’s historic series.
        - pct_change : float
            Most recent year-on-year % change.
        - weighted_pct_change : float
            Weighted average of the last 3 years.

    Returns
    -------
    str
        The recommended forecasting model name:
        - "Option 1: Crude Average"
        - "Option 2: Last-Year Only"
        - "Option 3: Weighted Last 3"
        - "Option 4: Linear Trend"

    Notes
    -----
    - High volatility (> 30), strong trend (|slope| > 5), or >= 2 missing years
      leads to a Linear Trend recommendation.
    - Stable series (volatility <= 10 and no missing years) default to Weighted Last 3,
      unless last-year shock > 25 percentage points.
    - Moderate volatility LAs (10 < vol <= 30) prefer Weighted Last 3 unless
      slope indicates a meaningful trend.
    """
    vol = row["volatility_proxy"]
    slope = row["linear_slope_per_year"]
    missing = row["missing_years"]
    last_yoy = row["pct_change"]
    weighted3 = row["weighted_pct_change"]

    # ---------- OPTION 4: Linear Trend ----------
    # High volatility OR strong trend OR many missing years
    if (
        (not pd.isna(vol) and vol > 30)
        or (not pd.isna(slope) and abs(slope) > 5)
        or (missing >= 2)
    ):
        return "Option 4: Linear Trend"

    # ---------- OPTION 3: Weighted Last 3 ----------
    # Stable behaviour
    if (not pd.isna(vol) and vol <= 10) and missing == 0:
        # but check if last-year shock is large → then Option 2
        if (
            pd.notna(last_yoy)
            and pd.notna(weighted3)
            and abs(last_yoy - weighted3) > 25
        ):
            return "Option 2: Last-Year Only"
        return "Option 3: Weighted Last 3"

    # ---------- Moderate Behaviour ----------
    if (not pd.isna(vol)) and (10 < vol <= 30):
        if not pd.isna(slope) and abs(slope) > 3:
            return "Option 4: Linear Trend"
        else:
            return "Option 3: Weighted Last 3"

    # ---------- OPTION 1: fallback ----------
    return "Option 1: Crude Average"



def choose_pct(row):
    """
    Select the appropriate forecast percentage for a single LA based on the
    recommended forecasting method.

    This function takes the model recommendation produced by
    ``recommend_model_row`` and returns the corresponding percentage value
    from the row. It acts as a dispatcher between the four forecasting
    approaches, ensuring that the chosen final YoY % value is consistent with
    the selected model.

    Parameters
    ----------
    row : pandas.Series
        A row containing:
        - recommended_method : str
            One of the four forecasting model names returned by
            ``recommend_model_row``.
        - avg_pct_change : float
            Crude average YoY %.
        - pct_change : float
            Most recent YoY % value.
        - weighted_pct_change : float
            Weighted average of the last three years.
        - linear_proj_next_year : float
            One-step-ahead projection from the linear trend model.

    Returns
    -------
    float
        The selected YoY % value corresponding to the recommended model.
        Returns NaN if none of the expected model names match.

    Notes
    -----
    - The function assumes ``recommended_method`` contains one of the values:
      "Option 1: Crude Average",
      "Option 2: Last-Year Only",
      "Option 3: Weighted Last 3",
      "Option 4: Linear Trend".
    - Missing or unexpected model names result in a NaN output.
    """
    m = row["recommended_method"]
    if m == "Option 1: Crude Average":
        return row["avg_pct_change"]
    if m == "Option 2: Last-Year Only":
        return row["pct_change"]
    if m == "Option 3: Weighted Last 3":
        return row["weighted_pct_change"]
    if m == "Option 4: Linear Trend":
        return row["linear_proj_next_year"]
    return np.nan





In [ ]:
#Data cleaning from SEN2 tables assessments.csv and requests.csv. Only importing data cols needed.
requests = requests[
    requests["breakdown_topic"] == "All requests for EHC needs assessments"
]
requests = requests[requests["old_la_code"].notna()]
# requests_region = requests[["time_period","region_name","requests_received_in_year"]]
requests_la = requests[
    [
        "time_period",
        "old_la_code",
        "requests_received_in_year",
        "requests_decided_not_to_assess",
    ]
]
requests_la["requests_received_in_year"] = pd.to_numeric(
    requests_la["requests_received_in_year"], errors="coerce"
)
requests_la["requests_decided_not_to_assess"] = pd.to_numeric(
    requests_la["requests_decided_not_to_assess"], errors="coerce"
)
requests_la["request_not_to_assess_pc"] = (
    requests_la["requests_decided_not_to_assess"]
    / requests_la["requests_received_in_year"]
    * 100
)

assessments = assessments[assessments["breakdown_topic"] == "All EHC needs assessments"]
assessments = assessments[assessments["old_la_code"].notna()]
assessments_la = assessments[["time_period", "old_la_code", "assess_not_issued_pc"]]

requests_la = requests_la.merge(
    assessments_la, on=["old_la_code", "time_period"], how="left"
)

requests_la["assess_not_issued_pc"] = (
    requests_la["assess_not_issued_pc"].replace("x", np.nan).astype(float)
)

# requests_la.head(5)
# assessments_la.head(10)

In [4]:
# Option 1
# Calculating a crude-average (YoY increase) forecast of requests per LA given SEN2 published data.
requests_la = requests_la.sort_values(["old_la_code", "time_period"])
requests_la["requests_ffill"] = requests_la.groupby("old_la_code")[
    "requests_received_in_year"
].ffill()
# the above line is needed to deal with missing years of data requests
requests_la["pct_change"] = (
    requests_la.groupby("old_la_code")["requests_ffill"].pct_change() * 100
)
avg_pct_req_la_crude = (
    requests_la.groupby("old_la_code")["pct_change"]
    .mean(skipna=True)
    .reset_index()
    .rename(columns={"pct_change": "avg_pct_change"})
)

# Calculating a crude-average forecast for % of no to assess and no to issue.
avg_R2A = (
    requests_la.groupby("old_la_code")["request_not_to_assess_pc"]
    .mean(skipna=True)
    .reset_index()
    .rename(columns={"request_not_to_assess_pc": "avg_R2A_pct"})
)
avg_R2I = (
    requests_la.groupby("old_la_code")["assess_not_issued_pc"]
    .mean(skipna=True)
    .reset_index()
    .rename(columns={"assess_not_issued_pc": "avg_R2I_pct"})
)

avg_pct_req_la_crude = avg_pct_req_la_crude.merge(
    avg_R2A, on="old_la_code", how="left"
).merge(avg_R2I, on="old_la_code", how="left")

# avg_pct_req_la_crude.head(5)
# requests_la.head(10)

In [5]:
# Option 2
# Calculating a most recent year forecast, based on the most recent SEN2 request data and most recent % difference behaviour, most recent % of no to assess and no to plan.
requests_la = requests_la.sort_values(["old_la_code", "time_period"])
latest_pct = requests_la.groupby("old_la_code").tail(1)[
    [
        "old_la_code",
        "time_period",
        "pct_change",
        "request_not_to_assess_pc",
        "assess_not_issued_pc",
    ]
]

In [6]:
# Option 3
# Weighted Average (last 3 years only)

weighted_pct_avg_diff_la = (
    requests_la.groupby("old_la_code")["pct_change"]
    .apply(weighted_avg_last3)
    .reset_index(name="weighted_pct_change")
)

r2a_weighted = (
    requests_la.groupby("old_la_code")["request_not_to_assess_pc"]
    .apply(weighted_avg_last3)
    .reset_index(name="weighted_pct_R2A")
)

r2i_weighted = (
    requests_la.groupby("old_la_code")["assess_not_issued_pc"]
    .apply(weighted_avg_last3)
    .reset_index(name="weighted_pct_R2I")
)


weighted_pct_avg_diff_la = weighted_pct_avg_diff_la.merge(
    r2a_weighted, on="old_la_code", how="left"
).merge(r2i_weighted, on="old_la_code", how="left")

#weighted_pct_avg_diff_la.head(5)

In [ ]:
# Option 4 (Linear projection)
#Tame outliers before conducting linear forecasting.
# Tame YoY%
tamed_pct = (
    requests_la.groupby("old_la_code", group_keys=True)
    .apply(
        lambda g: tame_outliers(
            g,
            value_col="pct_change",
            out_col="pct_tamed",
            k=2.5,
            global_clip=(-100, 200),
        )
    )
    .reset_index(level=0)[  # <-- CRITICAL FIX
        ["old_la_code", "time_period", "pct_tamed"]
    ]
)

# Tame Refuse to Assess and Refuse to Issue %
tamed_r2a = (
    requests_la.groupby("old_la_code", group_keys=True)
    .apply(
        lambda g: tame_outliers(
            g,
            value_col="request_not_to_assess_pc",
            out_col="r2a_tamed",
            k=2.5,
            global_clip=(0, 100),
        )
    )
    .reset_index(level=0)[["old_la_code", "time_period", "r2a_tamed"]]
)


tamed_r2i = (
    requests_la.groupby("old_la_code", group_keys=True)
    .apply(
        lambda g: tame_outliers(
            g,
            value_col="assess_not_issued_pc",
            out_col="r2i_tamed",
            k=2.5,
            global_clip=(0, 100),
        )
    )
    .reset_index(level=0)[  # <-- CRITICAL FIX
        ["old_la_code", "time_period", "r2i_tamed"]
    ]
)


requests_tamed = (
    requests_la.merge(tamed_pct, on=["old_la_code", "time_period"], how="left")
    .merge(tamed_r2a, on=["old_la_code", "time_period"], how="left")
    .merge(tamed_r2i, on=["old_la_code", "time_period"], how="left")
)


# Fits a linear regression for YoY%
trend_yoy = linear_trend_by_column(
    requests_tamed,
    value_col="pct_tamed",
    latest_aux_cols=("requests_received_in_year",),
)

# Forecast Refuse to Assess
trend_r2a = linear_trend_by_column(
    requests_tamed, value_col="r2a_tamed", latest_aux_cols=()
)
trend_r2a = trend_r2a.rename(
    columns={
        "latest_value": "r2a_latest",
        "linear_slope_per_year": "r2a_slope",
        "linear_intercept": "r2a_intercept",
        "linear_proj_next_year": "r2a_linear_proj",
    }
)
trend_r2a["r2a_linear_proj"] = trend_r2a["r2a_linear_proj"].clip(0, 100)

# Forecast Refuse to Issue
trend_r2i = linear_trend_by_column(
    requests_tamed, value_col="r2i_tamed", latest_aux_cols=()
)
trend_r2i = trend_r2i.rename(
    columns={
        "latest_value": "r2i_latest",
        "linear_slope_per_year": "r2i_slope",
        "linear_intercept": "r2i_intercept",
        "linear_proj_next_year": "r2i_linear_proj",
    }
)
trend_r2i["r2i_linear_proj"] = trend_r2i["r2i_linear_proj"].clip(0, 100)

# merge into one single data frame
trend_df = trend_yoy.merge(trend_r2a, on="old_la_code", how="left").merge(
    trend_r2i, on="old_la_code", how="left"
)

#print(trend_df.columns.tolist())

In [8]:
# Suggestion to LA regarding which forecasting model to use based on their LA data behaviour.
# Create master table with all 4 forecasting method for each LA:

forecast_all = avg_pct_req_la_crude.copy()
forecast_all = forecast_all.merge(latest_pct, on="old_la_code", how="left")
forecast_all = forecast_all.merge(
    weighted_pct_avg_diff_la, on="old_la_code", how="left"
)
forecast_all = forecast_all.merge(trend_df, on="old_la_code", how="left")

#print(forecast_all.columns.tolist())

# Volatility proxy across available summary %s
forecast_all["volatility_proxy"] = forecast_all.apply(row_std_proxy, axis=1)

# Recent shock measure: how different last year is vs the 3yr weighted
forecast_all["recent_shock_abs"] = (
    forecast_all["pct_change"] - forecast_all["weighted_pct_change"]
).abs()

# Trend strength (already have slope)
forecast_all["trend_strength"] = forecast_all["linear_slope_per_year"].abs()

# If you truly cannot bring missing years from source, set to 0 (or leave NaN)
forecast_all["missing_years"] = 0  # Placeholder if you can’t compute from requests_la

forecast_all["recommended_method"] = forecast_all.apply(recommend_model_row, axis=1)

forecast_all["chosen_pct_change"] = forecast_all.apply(choose_pct, axis=1)

forecast_all = forecast_all[
    [
        "old_la_code",
        "volatility_proxy",
        "recent_shock_abs",
        "trend_strength",
        "missing_years",
        "recommended_method",
        "chosen_pct_change",
    ]
]

/usr/local/python/3.12.1/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1845: RuntimeWarning: invalid value encountered in subtract
  np.subtract(arr, avg, out=arr, casting='unsafe', where=where)


In [ ]:
# with pd.ExcelWriter(
#     "SEND Performance and Capacity Tool - Calculation.xlsx", engine="xlsxwriter"
# ) as writer:
#     avg_pct_req_la_crude.to_excel(writer, sheet_name="Crude Average", index=False)
#     latest_pct.to_excel(writer, sheet_name="Latest Average", index=False)
#     weighted_pct_avg_diff_la.to_excel(
#         writer, sheet_name="Weighted Average", index=False
#     )
#     trend_df.to_excel(writer, sheet_name="Linear Trend", index=False)
#     forecast_all.to_excel(writer, sheet_name="Suggested Forecast", index=False)